In [1]:
pip install gensim

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.


Question 1 : 

Modify the codes for both TF-IDF & Word2Vec vectorizer by adding text preprocessing steps.  
Do the Purity differ when applying text preprocessing before vectorization? 

In [1]:
# TF-IDF

import pandas as pd
import re 
import string 
import nltk
import numpy as np 
from sklearn.cluster import KMeans 
from sklearn.feature_extraction.text import TfidfVectorizer 
from tabulate import tabulate 
from collections import Counter
from nltk.corpus import stopwords 
 
stop_words = stopwords.words('english') 
more_stopwords = ['u', 'im', 'c'] 
stop_words = stop_words + more_stopwords

stemmer = nltk.SnowballStemmer("english") 

def preprocess(text): 
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text) 
    text = re.sub(r'http\S+\s*\S+', '', text) 
    text = re.sub(r'www\.\S+', '', text) 
    text = re.sub(r'<.*?>', '', text) 
    text = re.sub(r'[^\w\s]', '', text) 
    text = re.sub(r'\b\w*\d\w*\b', '', text) 
    words = [stemmer.stem(w) for w in text.split() if w and w not in stop_words]
    return " ".join(words)

dataset = ["I love playing football on the weekends", 
           "I enjoy hiking and camping in the mountains", 
           "I like to read books and watch movies", 
           "I prefer playing video games over sports", 
           "I love listening to music and going to concerts"] 

cleaned_dataset = [preprocess(doc) for doc in dataset] 

vectorizer = TfidfVectorizer() 
X = vectorizer.fit_transform(cleaned_dataset)

k = 2
km = KMeans(n_clusters=k) 
km.fit(X) 
 
y_pred = km.predict(X) 
 
table_data = [["Document", "Predicted Cluster"]] 
table_data.extend([[doc, cluster] for doc, cluster in zip(cleaned_dataset, y_pred)]) 
print(tabulate(table_data, headers="firstrow"))

print("\nTop terms per cluster:") 
order_centroids = km.cluster_centers_.argsort()[:, ::-1] 
terms = vectorizer.get_feature_names_out() 
for i in range(k): 
    print("Cluster %d:" % i) 
    for ind in order_centroids[i, :10]: 
        print(' %s' % terms[ind]) 
    print()

total_samples = len(y_pred) 
cluster_label_counts = [Counter(y_pred)] 
purity = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples 
print("Purity:", purity) 


Document                        Predicted Cluster
----------------------------  -------------------
love play footbal weekend                       0
enjoy hike camp mountain                        1
like read book watch movi                       1
prefer play video game sport                    0
love listen music go concert                    1

Top terms per cluster:
Cluster 0:
 play
 footbal
 weekend
 video
 sport
 game
 prefer
 love
 watch
 read

Cluster 1:
 mountain
 hike
 enjoy
 camp
 go
 listen
 music
 concert
 movi
 watch

Purity: 0.6


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


In [2]:
# WORD2VEC

import pandas as pd
import re 
import nltk
import numpy as np 
from sklearn.cluster import KMeans 
from gensim.models import Word2Vec 
from tabulate import tabulate 
from collections import Counter
from nltk.corpus import stopwords 

stop_words = stopwords.words('english') 
more_stopwords = ['u', 'im', 'c'] 
stop_words = stop_words + more_stopwords

stemmer = nltk.SnowballStemmer("english") 

def preprocess(text): 
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text) 
    text = re.sub(r'http\S+\s*\S+', '', text) 
    text = re.sub(r'www\.\S+', '', text) 
    text = re.sub(r'<.*?>', '', text) 
    text = re.sub(r'[^\w\s]', '', text) 
    text = re.sub(r'\b\w*\d\w*\b', '', text) 
    words = [stemmer.stem(w) for w in text.split() if w and w not in stop_words]
    return " ".join(words)

dataset = ["I love playing football on the weekends", 
           "I enjoy hiking and camping in the mountains", 
           "I like to read books and watch movies", 
           "I prefer playing video games over sports", 
           "I love listening to music and going to concerts"] 

cleaned_dataset = [preprocess(doc) for doc in dataset]

tokenized_dataset = [doc.split() for doc in cleaned_dataset] 
word2vec_model = Word2Vec(sentences=tokenized_dataset, vector_size=100, window=5, min_count=1, workers=4) 

X = np.array([
    np.mean([word2vec_model.wv[word] for word in doc.split() if word in word2vec_model.wv], axis=0) 
    for doc in cleaned_dataset
]) 

k = 2  
km = KMeans(n_clusters=k) 
km.fit(X) 
 
y_pred = km.predict(X) 
 
table_data = [["Document", "Predicted Cluster"]] 
table_data.extend([[doc, cluster] for doc, cluster in zip(cleaned_dataset, y_pred)]) 
print(tabulate(table_data, headers="firstrow"))

total_samples = len(y_pred) 
cluster_label_counts = [Counter(y_pred)] 
purity = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples 
print("Purity:", purity)

Document                        Predicted Cluster
----------------------------  -------------------
love play footbal weekend                       1
enjoy hike camp mountain                        1
like read book watch movi                       1
prefer play video game sport                    1
love listen music go concert                    0
Purity: 0.8


Answer: Yes but only for word2vec

Question 2 : 

Perform text clustering on 'customer_complaints_1.csv' dataset, specifically the Text column. 

In [6]:
# TF-IDF

import pandas as pd
import re
import string
import nltk
import numpy as np 
from nltk.corpus import stopwords
from sklearn.cluster import KMeans 
from sklearn.feature_extraction.text import TfidfVectorizer 
from tabulate import tabulate 
from collections import Counter

df = pd.read_csv("customer_complaints_1.csv",encoding="ISO-8859-1") 

stop_words = stopwords.words('english') 
more_stopwords = ['u', 'im', 'c'] 
stop_words = stop_words + more_stopwords

stemmer = nltk.SnowballStemmer("english") 

def preprocess(text): 
    text = text.lower()  # Convert text to lowercase 
    text = re.sub(r'\[.*?\]', '', text)  # Remove text within square brackets 
    text = re.sub(r'http\S+\s*\S+', '', text)  # Remove URLs starting with http 
    text = re.sub(r'www\.\S+', '', text)  # Remove URLs starting with www 
    text = re.sub(r'<.*?>', '', text)  # Remove HTML tags 
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation 
    text = re.sub(r'\b\w*\d\w*\b', '', text)  # Remove words containing numbers 
    text = ' '.join(word for word in text.split(' ') if word not in stop_words) #remove stopwords 
    text = ' '.join(stemmer.stem(word) for word in text.split(' ')) #stemming 
    return text

df['text_clean'] = df['text'].apply(preprocess) 
print(df[['text','text_clean']])

vectorizer = TfidfVectorizer() 
X = vectorizer.fit_transform(df['text_clean']) 

k = 2 
km = KMeans(n_clusters=k) 
km.fit(X) 
 
y_pred = km.predict(X) 
 
table_data = [["Document", "Predicted Cluster"]] 
table_data.extend([[doc, cluster] for doc, cluster in zip(df['text_clean'], y_pred)]) 
print(tabulate(table_data, headers="firstrow"))

print("\nTop terms per cluster:") 
order_centroids = km.cluster_centers_.argsort()[:, ::-1] 
terms = vectorizer.get_feature_names_out() 
for i in range(k): 
    print("Cluster %d:" % i) 
    for ind in order_centroids[i, :10]: 
        print(' %s' % terms[ind]) 
    print()

total_samples = len(y_pred) 
cluster_label_counts = [Counter(y_pred)] 
purity = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples 
print("Purity:", purity) 


                                                 text  \
0   I used to love Comcast. Until all these consta...   
1   I'm so over Comcast! The worst internet provid...   
2   If I could give them a negative star or no sta...   
3   I've had the worst experiences so far since in...   
4   Check your contract when you sign up for Comca...   
5   Thank God. I am changing to Dish. They gave me...   
6   I Have been a long time customer and only have...   
7   There is a malfunction on the DVR manager whic...   
8   Charges overwhelming. Comcast service rep was ...   
9   I have had cable, DISH, and U-verse, etc. in t...   
10  Had them from 2014 to now. I'd tell new custom...   
11  Disappointed. I have been a Comcast/Xfinity cu...   
12  These people are unethical and disturbing obli...   
13  Unplanned, unexpected, all day outages, rude s...   
14  BE WARNED. You will have 10$ hidden fees when ...   
15  Had Comcast. Overall the terrible experience e...   
16  When I called the infinity 

In [14]:
# WORD2VEC

import pandas as pd
import re
import string
import nltk
import numpy as np

from nltk.corpus import stopwords
from sklearn.cluster import KMeans 
from gensim.models import Word2Vec 
from tabulate import tabulate 
from collections import Counter 

df = pd.read_csv("customer_complaints_1.csv",encoding="ISO-8859-1") 

stop_words = stopwords.words('english') 
more_stopwords = ['u', 'im', 'c'] 
stop_words = stop_words + more_stopwords

stemmer = nltk.SnowballStemmer("english") 

def preprocess(text): 
    text = text.lower()  # Convert text to lowercase 
    text = re.sub(r'\[.*?\]', '', text)  # Remove text within square brackets 
    text = re.sub(r'http\S+\s*\S+', '', text)  # Remove URLs starting with http 
    text = re.sub(r'www\.\S+', '', text)  # Remove URLs starting with www 
    text = re.sub(r'<.*?>', '', text)  # Remove HTML tags 
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation 
    text = re.sub(r'\b\w*\d\w*\b', '', text)  # Remove words containing numbers 
    text = ' '.join(word for word in text.split(' ') if word not in stop_words) #remove stopwords 
    text = ' '.join(stemmer.stem(word) for word in text.split(' ')) #stemming 
    return text

df['text_clean'] = df['text'].apply(preprocess) 
print(df[['text','text_clean']])

tokenized_dataset = [doc.split() for doc in  df['text_clean']] 
word2vec_model = Word2Vec(sentences=tokenized_dataset, vector_size=100, window=5, min_count=1, workers=4) 

X = np.array([np.mean([word2vec_model.wv[word] for word in doc.split() if word in word2vec_model.wv], axis=0) for doc in df['text_clean']])

k = 2  
km = KMeans(n_clusters=k) 
km.fit(X)

y_pred = km.predict(X) 

table_data = [["Document", "Predicted Cluster"]] 
table_data.extend([[doc, cluster] for doc, cluster in zip(df['text_clean'], y_pred)]) 
print(tabulate(table_data, headers="firstrow"))

total_samples = len(y_pred) 
cluster_label_counts = [Counter(y_pred)] 
purity = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples 
print("Purity:", purity)

                                                 text  \
0   I used to love Comcast. Until all these consta...   
1   I'm so over Comcast! The worst internet provid...   
2   If I could give them a negative star or no sta...   
3   I've had the worst experiences so far since in...   
4   Check your contract when you sign up for Comca...   
5   Thank God. I am changing to Dish. They gave me...   
6   I Have been a long time customer and only have...   
7   There is a malfunction on the DVR manager whic...   
8   Charges overwhelming. Comcast service rep was ...   
9   I have had cable, DISH, and U-verse, etc. in t...   
10  Had them from 2014 to now. I'd tell new custom...   
11  Disappointed. I have been a Comcast/Xfinity cu...   
12  These people are unethical and disturbing obli...   
13  Unplanned, unexpected, all day outages, rude s...   
14  BE WARNED. You will have 10$ hidden fees when ...   
15  Had Comcast. Overall the terrible experience e...   
16  When I called the infinity 